# Suicide Rate Prediction - Training Notebook

Exploration des données et entraînement de 3 modèles de régression.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from config import DATA_DIR, MODELS_DIR, PLOTS_DIR
from data import load_dataset_split, FEATURES, TARGET

print('Imports OK')

## 1. Chargement et exploration des données

In [ ]:
# Load raw data for exploration
df_raw = pd.read_csv(DATA_DIR / 'master.csv')
print(f'Shape: {df_raw.shape}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head()

In [ ]:
df_raw.describe()

In [ ]:
df_raw.isnull().sum()

## 2. Visualisations exploratoires

In [ ]:
# Distribution of suicide rate
fig, ax = plt.subplots(figsize=(10, 5))
df_raw['suicides/100k pop'].hist(bins=50, ax=ax, edgecolor='black')
ax.set_title('Distribution du taux de suicide pour 100k habitants')
ax.set_xlabel('suicides/100k pop')
ax.set_ylabel('Frequence')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'distribution_target.png', dpi=150)
plt.show()
print('Plot saved: distribution_target.png')

In [ ]:
# Suicide rate by sex
fig, ax = plt.subplots(figsize=(8, 5))
df_raw.groupby('sex')['suicides/100k pop'].mean().plot(kind='bar', ax=ax, color=['#FF6B6B', '#4ECDC4'])
ax.set_title('Taux moyen de suicide par sexe')
ax.set_ylabel('suicides/100k pop')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'suicide_by_sex.png', dpi=150)
plt.show()
print('Plot saved: suicide_by_sex.png')

In [ ]:
# Suicide rate by age group
age_order = ['5-14 years', '15-24 years', '25-34 years', '35-54 years', '55-74 years', '75+ years']
fig, ax = plt.subplots(figsize=(10, 5))
df_raw.groupby('age')['suicides/100k pop'].mean().reindex(age_order).plot(kind='bar', ax=ax, color='#45B7D1')
ax.set_title('Taux moyen de suicide par tranche d\'age')
ax.set_ylabel('suicides/100k pop')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'suicide_by_age.png', dpi=150)
plt.show()
print('Plot saved: suicide_by_age.png')

In [ ]:
# Temporal evolution
fig, ax = plt.subplots(figsize=(12, 5))
yearly = df_raw.groupby('year')['suicides/100k pop'].mean()
yearly.plot(ax=ax, marker='o', color='#6C5CE7')
ax.set_title('Evolution temporelle du taux moyen de suicide')
ax.set_ylabel('suicides/100k pop')
ax.set_xlabel('Annee')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'temporal_evolution.png', dpi=150)
plt.show()
print('Plot saved: temporal_evolution.png')

In [ ]:
# Top 10 countries
fig, ax = plt.subplots(figsize=(12, 6))
top10 = df_raw.groupby('country')['suicides/100k pop'].mean().sort_values(ascending=True).tail(10)
top10.plot(kind='barh', ax=ax, color='#E17055')
ax.set_title('Top 10 pays par taux moyen de suicide')
ax.set_xlabel('suicides/100k pop')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'top10_countries.png', dpi=150)
plt.show()
print('Plot saved: top10_countries.png')

## 3. Chargement du dataset preprocesse

In [ ]:
X_train, X_test, y_train, y_test = load_dataset_split()
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape: {y_test.shape}')
print(f'\nFeatures: {list(X_train.columns)}')

In [ ]:
# Correlation heatmap on training data
fig, ax = plt.subplots(figsize=(10, 8))
train_corr = pd.concat([X_train, y_train], axis=1)
sns.heatmap(train_corr.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Matrice de correlation des features')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'correlation_heatmap.png', dpi=150)
plt.show()
print('Plot saved: correlation_heatmap.png')

## 4. Entrainement des modeles

In [ ]:
# Model 1: Linear Regression
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)
print(f'Linear Regression - MAE: {mean_absolute_error(y_test, y_pred_lr):.4f}, R2: {r2_score(y_test, y_pred_lr):.4f}')
joblib.dump(pipe_lr, MODELS_DIR / 'linear_reg.joblib')
print('Saved: linear_reg.joblib')

In [ ]:
# Model 2: Random Forest
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42))
])
pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)
print(f'Random Forest - MAE: {mean_absolute_error(y_test, y_pred_rf):.4f}, R2: {r2_score(y_test, y_pred_rf):.4f}')
joblib.dump(pipe_rf, MODELS_DIR / 'random_forest.joblib')
print('Saved: random_forest.joblib')

In [ ]:
# Model 3: Gradient Boosting
pipe_gb = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42))
])
pipe_gb.fit(X_train, y_train)
y_pred_gb = pipe_gb.predict(X_test)
print(f'Gradient Boosting - MAE: {mean_absolute_error(y_test, y_pred_gb):.4f}, R2: {r2_score(y_test, y_pred_gb):.4f}')
joblib.dump(pipe_gb, MODELS_DIR / 'gradient_boosting.joblib')
print('Saved: gradient_boosting.joblib')

## 5. Comparaison visuelle des modeles

In [ ]:
# Predictions vs Actual for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_preds = [
    ('Linear Regression', y_pred_lr),
    ('Random Forest', y_pred_rf),
    ('Gradient Boosting', y_pred_gb),
]

for ax, (name, y_pred) in zip(axes, models_preds):
    ax.scatter(y_test, y_pred, alpha=0.3, s=10)
    lims = [0, max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', lw=1)
    ax.set_title(name)
    ax.set_xlabel('Valeurs reelles')
    ax.set_ylabel('Predictions')
    ax.set_xlim(lims)
    ax.set_ylim(lims)

plt.suptitle('Predictions vs Valeurs reelles', fontsize=14)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'predictions_vs_actual.png', dpi=150)
plt.show()
print('Plot saved: predictions_vs_actual.png')

In [ ]:
# Residual distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, y_pred) in zip(axes, models_preds):
    residuals = y_test.values - y_pred
    ax.hist(residuals, bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(f'{name} - Residus')
    ax.set_xlabel('Residus')
    ax.set_ylabel('Frequence')
    ax.axvline(0, color='red', linestyle='--')

plt.suptitle('Distribution des residus', fontsize=14)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'residuals_distribution.png', dpi=150)
plt.show()
print('Plot saved: residuals_distribution.png')

In [ ]:
# Feature importance for Random Forest
rf_model = pipe_rf.named_steps['model']
feat_imp = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.plot(kind='barh', ax=ax, color='#00B894')
ax.set_title('Importance des features (Random Forest)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'feature_importance_rf.png', dpi=150)
plt.show()
print('Plot saved: feature_importance_rf.png')

In [ ]:
# Summary metrics comparison
from metrics import compute_metrics

results = []
for name, y_pred in models_preds:
    m = compute_metrics(y_test, y_pred)
    m['model'] = name
    results.append(m)

results_df = pd.DataFrame(results)[['model', 'mae', 'rmse', 'r2', 'mape']]
print(results_df.to_string(index=False))
print('\nEntrainement termine! Les modeles sont sauvegardes dans models/')